# TackleNet usage notebook

TackleNet contains 737 American football practice clips with six Standard
Assessment of Tackling Technique (SATT) scores and a zero-based first point
of contact (FPOC) frame index. This notebook checks the annotations and
frozen split, plots dataset statistics, and demonstrates the 32-frame
contact window on locally available videos.

SATT3 grades 0–1 map to risky technique and grades 2–3 map to safe technique.
The sum of the six SATT scores is descriptive. The binary label comes from
SATT3.

Data are available under CC BY 4.0. Benchmark code is released under the MIT
License.

## 1. Local setup

Install the packages in `requirements.txt` from the repository or review
folder, then open the notebook in Jupyter and select **Run All**. A standalone
environment needs NumPy, pandas, Matplotlib, scikit-learn, OpenCV, and
Jupyter. The notebook reads local files after setup.

- **Extracted review or dataset folder.** Keep
  `annotations_satt_fpoc.csv` and `videos/` together. An optional
  `splits_frozen.csv` can sit beside the annotation file.
- **Repository.** Open from the root or `notebooks/`. The notebook reads
  the CSVs in `splits/` and looks for clips in `videos/` or
  `tacklenet_data/videos/`. With only the CSVs it runs the annotation
  and split sections.
- **Separate video subset.** Set `VIDEO_DIR` to its video folder. The
  MP4 filenames must match the full annotation CSV. Set `DATA_ROOT`
  if the annotation files are in a different folder from the notebook.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

In [ ]:
DATA_ROOT = None          # Example: Path("path/to/extracted_folder")
VIDEO_DIR = None          # Relative to DATA_ROOT, or an absolute path
PREVIEW_CLIP = None       # Optional exact MP4 filename for the contact-window preview
CHECK_ALL_VIDEOS = False  # Enable to decode every matched MP4 and check its frame range

if DATA_ROOT is None:
    root = Path.cwd()
    if root.name == "notebooks":
        root = root.parent
else:
    root = Path(DATA_ROOT).expanduser().resolve()

annotation_candidates = (
    root / "annotations_satt_fpoc.csv",
    root / "splits" / "annotations_satt_fpoc.csv",
)
ANNOT_PATH = next((path for path in annotation_candidates if path.is_file()), None)
if ANNOT_PATH is None:
    raise FileNotFoundError(
        "Set DATA_ROOT to the folder containing annotations_satt_fpoc.csv "
        "or the repository root containing splits/annotations_satt_fpoc.csv."
    )

split_candidates = (ANNOT_PATH.with_name("splits_frozen.csv"), root / "splits_frozen.csv")
SPLITS_PATH = next((path for path in split_candidates if path.is_file()), None)

if VIDEO_DIR is None:
    video_candidates = (root / "videos", root / "tacklenet_data" / "videos")
    VIDEOS_DIR = next((path for path in video_candidates if path.is_dir()), None)
else:
    VIDEOS_DIR = root / Path(VIDEO_DIR).expanduser()
    if not VIDEOS_DIR.is_dir():
        raise FileNotFoundError("Set VIDEO_DIR to an existing folder containing the MP4s.")
available_videos = sorted(
    path.name for path in VIDEOS_DIR.glob("*.mp4") if path.is_file()
) if VIDEOS_DIR else []

print(f"Annotations: {ANNOT_PATH.relative_to(root)}")
print(f"Local videos: {len(available_videos)}")
print("Frozen split: " + ("read from CSV" if SPLITS_PATH else "recreated from annotations"))

## 2. Annotation checks and dataset statistics

These tables and plots describe the full annotation table. The preview
video folder can contain a subset of the annotated clips.

In [ ]:
ann = pd.read_csv(ANNOT_PATH)

REQUIRED_COLS = [
    "clip_id", "clip_filename", "Session_ID", "fpoc_frame",
    "satt1", "satt2", "satt3_score", "satt4", "satt5", "satt6",
    "satt_sum", "satt3_risk_binary",
]
missing_cols = [c for c in REQUIRED_COLS if c not in ann.columns]
assert not missing_cols, f"Missing annotation columns: {missing_cols}"

assert len(ann) == 737,                "Expected 737 rows"
assert ann["clip_id"].is_unique,       "clip_id is not unique"
assert ann["clip_filename"].is_unique, "clip_filename is not unique"
assert ann["Session_ID"].notna().all(), "Session_ID contains missing values"

# SATT component ranges
SATT_COLS = ["satt1", "satt2", "satt3_score", "satt4", "satt5", "satt6"]
for c in SATT_COLS:
    assert ann[c].isin([0, 1, 2, 3]).all(), f"Unexpected values in {c}"

# satt_sum consistency
assert (ann["satt_sum"] == ann[SATT_COLS].sum(axis=1)).all(), (
    "satt_sum does not match the sum of SATT1–SATT6"
)

# Binary label derivation
assert ann["satt3_risk_binary"].isin([0, 1]).all(), "Unexpected values in satt3_risk_binary"
derived = ann["satt3_score"].isin([0, 1]).astype(int)
assert (derived == ann["satt3_risk_binary"]).all(), (
    "satt3_risk_binary does not match the satt3_score rule (0/1 -> risky, 2/3 -> safe)"
)

# fpoc_frame: zero-based, non-negative integer
assert ann["fpoc_frame"].notna().all(), "fpoc_frame contains missing values"
assert (ann["fpoc_frame"] >= 0).all(), "fpoc_frame contains negative values"
assert (ann["fpoc_frame"] == ann["fpoc_frame"].astype(int)).all(), (
    "fpoc_frame contains non-integer values (possible CSV float encoding)"
)
ann["fpoc_frame"] = ann["fpoc_frame"].astype(int)

print("All annotation integrity checks passed.")
print(f"  Clips          : {len(ann)}")
print(f"  Sessions       : {ann['Session_ID'].nunique()}")
print(f"  Risky (1)      : {ann['satt3_risk_binary'].sum()}")
print(f"  Safe  (0)      : {(ann['satt3_risk_binary'] == 0).sum()}")
print(f"  satt_sum range : {ann['satt_sum'].min()}–{ann['satt_sum'].max()}")
print(f"  FPOC range     : {ann['fpoc_frame'].min()}–{ann['fpoc_frame'].max()}")
display(ann.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
counts = ann["satt3_risk_binary"].value_counts().reindex([0, 1], fill_value=0)
labels = ["Safe (0)", "Risky (1)"]
colors = ["#4C72B0", "#DD8452"]
axes[0].bar(labels, counts.values, color=colors, width=0.4)
axes[0].set_title("Class Balance")
axes[0].set_ylabel("Clips")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha="center", fontsize=11)

# SATT3 score distribution
score_counts = ann["satt3_score"].value_counts().reindex([0, 1, 2, 3], fill_value=0)
bar_colors   = ["#DD8452", "#DD8452", "#4C72B0", "#4C72B0"]
axes[1].bar([str(s) for s in score_counts.index], score_counts.values,
            color=bar_colors, width=0.4)
axes[1].set_title("SATT3 Score Distribution (source of binary label)")
axes[1].set_xlabel("satt3_score")
axes[1].set_ylabel("Clips")
risky_p = mpatches.Patch(color="#DD8452", label="Risky (score 0–1)")
safe_p  = mpatches.Patch(color="#4C72B0", label="Safe (score 2–3)")
axes[1].legend(handles=[risky_p, safe_p])
for i, v in enumerate(score_counts.values):
    axes[1].text(i, v + 1, str(v), ha="center", fontsize=10)

plt.suptitle("TackleNet — Annotation Overview", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(ann["fpoc_frame"], bins=40, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.set_title("FPOC Frame Distribution")
ax.set_xlabel("fpoc_frame (0-indexed frame number within clip)")
ax.set_ylabel("Clips")
plt.tight_layout()
plt.show()

print(f"FPOC range  : {ann['fpoc_frame'].min()} – {ann['fpoc_frame'].max()}")
print(f"Median FPOC : {ann['fpoc_frame'].median():.0f}")

In [ ]:
# satt_sum is auxiliary: shown for description only
fig, ax = plt.subplots(figsize=(9, 3))
for label, color, name in [(0, "#4C72B0", "Safe"), (1, "#DD8452", "Risky")]:
    subset = ann[ann["satt3_risk_binary"] == label]["satt_sum"]
    ax.hist(subset, bins=18, alpha=0.7, color=color, edgecolor="white",
            linewidth=0.3, label=name)
ax.set_title("satt_sum Distribution by Risk Label  [auxiliary, not the label source]")
ax.set_xlabel("satt_sum (sum of SATT1–SATT6, range 0–18)")
ax.set_ylabel("Clips")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Clips per recording session, split by binary label
by_session = (
    ann.groupby(["Session_ID", "satt3_risk_binary"]).size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1], fill_value=0)
    .sort_index()
)
session_sizes = by_session.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 3))
x = by_session.index.astype(str)
ax.bar(x, by_session[0], color="#4C72B0", width=0.6, label="Safe (0)")
ax.bar(x, by_session[1], bottom=by_session[0], color="#DD8452", width=0.6, label="Risky (1)")
ax.axhline(session_sizes.mean(), color="gray", linestyle="--", linewidth=1,
           label=f"Mean ({session_sizes.mean():.1f})")
ax.set_title("Clips per Session")
ax.set_xlabel("Session_ID")
ax.set_ylabel("Clips")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Sessions    : {len(session_sizes)}")
print(f"Min / Max   : {session_sizes.min()} / {session_sizes.max()} clips per session")
print(f"Mean        : {session_sizes.mean():.1f} clips per session")

## 3. Verify the frozen split

The frozen benchmark uses the seed-42 stratified clip-level split with
515 training, 111 validation, and 111 test clips. A practice session can
appear in multiple partitions. Use `Session_ID` for separate
session-disjoint analyses.

The notebook compares its regenerated assignments with `splits_frozen.csv`
when that file is available. If the CSV is absent, it reconstructs the
assignments in memory from the annotations. It preserves the annotation
row order when applying the frozen split procedure.

In [ ]:
SPLIT_COLUMNS = ["clip_id", "clip_filename", "satt3_risk_binary", "split"]
EXPECTED_SPLITS = {"train": 515, "val": 111, "test": 111}


def recreate_frozen_split(annotations: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """Apply the frozen clip-level split procedure to the annotation rows."""
    source = annotations[SPLIT_COLUMNS[:-1]].copy()
    train, remaining = train_test_split(
        source, test_size=0.30, stratify=source["satt3_risk_binary"], random_state=seed,
    )
    validation, test = train_test_split(
        remaining, test_size=0.50, stratify=remaining["satt3_risk_binary"], random_state=seed,
    )
    split_rows = pd.concat(
        [train.assign(split="train"), validation.assign(split="val"), test.assign(split="test")],
        ignore_index=True,
    )
    return split_rows[SPLIT_COLUMNS].sort_values("clip_id").reset_index(drop=True)


recreated = recreate_frozen_split(ann)
if SPLITS_PATH is None:
    splits = recreated
    print("Recreated the frozen assignments in memory.")
else:
    splits = pd.read_csv(SPLITS_PATH)
    missing_columns = set(SPLIT_COLUMNS) - set(splits.columns)
    if missing_columns:
        raise ValueError(f"Missing columns in splits_frozen.csv: {sorted(missing_columns)}")
    splits = splits[SPLIT_COLUMNS].sort_values("clip_id").reset_index(drop=True)
    pd.testing.assert_frame_equal(splits, recreated)
    print("All assignments match splits_frozen.csv.")

assert splits["split"].value_counts().to_dict() == EXPECTED_SPLITS
merged = ann.merge(splits[["clip_id", "split"]], on="clip_id", validate="one_to_one")
assert len(merged) == len(ann)
summary = (
    merged.groupby("split")["satt3_risk_binary"]
    .agg(clips="size", risky="sum", safe=lambda values: (values == 0).sum())
    .reindex(["train", "val", "test"])
)
display(summary)

## 4. Inspect local clips

The preview uses one local clip per SATT3 grade when available. Set
`PREVIEW_CLIP` in Section 1 to choose the contact-window example,
or `CHECK_ALL_VIDEOS = True` to validate the range of each local
clip's contact annotation. Video decoding follows the stored frame order.
Preview frames are resized for display.

In [ ]:
def inspect_video(path: Path, contact: int) -> dict:
    """Count decoded frames and check that the contact annotation is in range."""
    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            raise ValueError(f"OpenCV could not open {path.name}.")
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n_frames = 0
        while cap.grab():
            n_frames += 1
    finally:
        cap.release()
    if not 0 <= contact < n_frames:
        raise ValueError(f"{path.name}: contact frame {contact} is outside {n_frames} frames.")
    return {
        "clip_filename": path.name,
        "decoded_frames": n_frames,
        "fps": round(fps, 2),
        "resolution": f"{width}×{height}",
        "fpoc_frame": contact,
    }


def read_frames(path: Path, indices, max_side: int = 640) -> dict:
    """Read the requested RGB frames without seeking."""
    wanted = {int(index) for index in indices}
    if not wanted:
        return {}
    cap = cv2.VideoCapture(str(path))
    frames = {}
    try:
        if not cap.isOpened():
            raise ValueError(f"OpenCV could not open {path.name}.")
        for index in range(max(wanted) + 1):
            if not cap.grab():
                break
            if index not in wanted:
                continue
            ok, bgr = cap.retrieve()
            if ok:
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                height, width = rgb.shape[:2]
                if max(height, width) > max_side:
                    scale = max_side / max(height, width)
                    size = (max(1, round(width * scale)), max(1, round(height * scale)))
                    rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)
                frames[index] = rgb
    finally:
        cap.release()
    missing = wanted - set(frames)
    if missing:
        raise ValueError(f"{path.name}: could not decode frames {sorted(missing)}.")
    return frames


def show_frame_grid(frames, titles, suptitle: str = "") -> None:
    """Display the selected frames side by side."""
    fig, axes = plt.subplots(1, len(frames), figsize=(2.2 * len(frames), 2.4))
    for axis, frame, title in zip(np.atleast_1d(axes), frames, titles):
        axis.imshow(frame)
        axis.set_title(title, fontsize=9)
        axis.axis("off")
    fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

In [ ]:
local = merged.loc[merged["clip_filename"].isin(available_videos)]
local = local.sort_values(["satt3_score", "clip_id"]).set_index("clip_filename")
unmatched = len(available_videos) - len(local)
if unmatched:
    print(f"Skipping {unmatched} MP4s with no matching annotation filename.")

if PREVIEW_CLIP is not None and PREVIEW_CLIP not in local.index:
    raise ValueError("PREVIEW_CLIP must match a local MP4 and its annotation filename.")

preview = local.drop_duplicates("satt3_score")
if PREVIEW_CLIP is not None:
    selected_grade = local.loc[PREVIEW_CLIP, "satt3_score"]
    preview = pd.concat([
        local.loc[[PREVIEW_CLIP]],
        preview.loc[preview["satt3_score"] != selected_grade],
    ])

video_info = {}
if local.empty:
    print("Add annotated MP4s or set VIDEO_DIR to see the video examples.")
else:
    check_rows = local if CHECK_ALL_VIDEOS else preview
    for filename, row in check_rows.iterrows():
        video_info[filename] = inspect_video(VIDEOS_DIR / filename, int(row["fpoc_frame"]))
    display(pd.DataFrame(video_info.values()).set_index("clip_filename"))
    display(preview[["Session_ID", "split", "fpoc_frame", *SATT_COLS, "satt3_risk_binary"]])

## 5. Extract the 32-frame contact window

The window includes 23 frames before contact, the contact frame, and
eight frames after contact. Contact stays at zero-based position 23.
Indices beyond the clip's boundaries repeat the nearest boundary frame.

In [ ]:
def fpoc_window_indices(fpoc_frame: int, n_frames: int, pre: int = 23, post: int = 8) -> np.ndarray:
    """Clip-frame indices of the FPOC window, clamped to [0, n_frames - 1]."""
    if n_frames <= 0:
        raise ValueError("Empty clip: no decoded frames.")
    if pre < 0 or post < 0:
        raise ValueError("pre and post must be non-negative.")
    if not 0 <= fpoc_frame < n_frames:
        raise ValueError(f"fpoc_frame={fpoc_frame} is outside the valid range [0, {n_frames - 1}].")

    idx = np.clip(np.arange(fpoc_frame - pre, fpoc_frame + post + 1), 0, n_frames - 1)
    assert len(idx) == pre + 1 + post
    assert idx[pre] == fpoc_frame, f"FPOC frame is not at window index {pre}."
    return idx


def extract_fpoc_window(clip: np.ndarray, fpoc_frame: int, pre: int = 23, post: int = 8) -> np.ndarray:
    """Reference implementation for a clip already decoded into memory, shape (T, H, W, C).

    Returns a (pre + 1 + post, H, W, C) array with the FPOC frame at index `pre`.
    """
    if clip is None or len(clip) == 0:
        raise ValueError("Empty clip: no decoded frames.")
    return clip[fpoc_window_indices(fpoc_frame, len(clip), pre, post)]


def load_fpoc_window(path: Path, fpoc_frame: int, n_frames: int,
                     pre: int = 23, post: int = 8, max_side: int = 640) -> np.ndarray:
    """Same window as extract_fpoc_window, decoding only the frames the window needs."""
    idx = fpoc_window_indices(fpoc_frame, n_frames, pre, post)
    frames = read_frames(path, np.unique(idx), max_side=max_side)
    return np.stack([frames[int(i)] for i in idx])

In [ ]:
if not preview.empty:
    filename = preview.index[0]
    example = video_info[filename]
    contact = example["fpoc_frame"]
    n_frames = example["decoded_frames"]
    indices = fpoc_window_indices(contact, n_frames)
    window = load_fpoc_window(VIDEOS_DIR / filename, contact, n_frames)
    print(f"{filename}: first contact at frame {contact}, window shape {window.shape}.")

    full_indices = np.linspace(0, n_frames - 1, 8, dtype=int)
    full_frames = read_frames(VIDEOS_DIR / filename, full_indices)
    show_frame_grid(
        [full_frames[index] for index in full_indices],
        [f"Frame {index}" for index in full_indices],
        f"{filename} across the clip",
    )

    positions = (0, 8, 16, 22, 23, 24, 31)
    titles = [
        f"Window {position}\nFrame {indices[position]}"
        + (" (contact)" if position == 23 else "")
        for position in positions
    ]
    show_frame_grid(
        [window[position] for position in positions],
        titles,
        f"{filename}: selected frames from the 32-frame contact window",
    )

## 6. Compare contact frames across grades

Each row shows a selected local clip around its annotated contact frame.
The fourth column shows first contact. A video subset is useful for
inspection. Benchmark evaluation uses the full designated test partition.

In [ ]:
if not preview.empty:
    offsets = (-23, -12, -4, 0, 8)
    fig, axes = plt.subplots(
        len(preview), len(offsets),
        figsize=(3 * len(offsets), 2.2 * len(preview)), squeeze=False,
    )
    for row_index, (filename, row) in enumerate(preview.iterrows()):
        contact = int(row["fpoc_frame"])
        n_frames = video_info[filename]["decoded_frames"]
        indices = np.clip(contact + np.array(offsets), 0, n_frames - 1)
        frames = read_frames(VIDEOS_DIR / filename, indices, max_side=480)
        for axis, offset, index in zip(axes[row_index], offsets, indices):
            axis.imshow(frames[index])
            axis.set_title("Contact" if offset == 0 else f"{offset:+d} frames", fontsize=8)
            axis.set_xticks([])
            axis.set_yticks([])
        axes[row_index, 0].set_ylabel(f"{filename}\nSATT3 grade {row['satt3_score']}")
    fig.tight_layout()
    plt.show()